
## Folder 06 — File 03: Foreign Key Relationship Analysis:


- Parent-child record distribution.
- Number of child records per parent.
- Parents without child records.
- Orphan reference percentages.
- Relationship-level health metrics.
- Dashboard-ready relationship reports.


- A **Foreign Key (FK)** is a column in **one table (child table)** that references a key in **another table (parent table)**.
- It establishes a relationship between `two tables` and helps maintain `Referential Integrity`.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from datetime import datetime
import uuid

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

RELATIONSHIP_REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.foreign_key_relationship_report"
)

CARDINALITY_REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.foreign_key_cardinality_report"
)

RUN_ID = str(uuid.uuid4())
CHECKED_AT = datetime.now()

print("Run ID:", RUN_ID)
print("Checked at:", CHECKED_AT)


## 3. Define Silver source tables

TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments",
    "inventory": f"{CATALOG}.{SCHEMA}.silver_inventory"
}

missing_tables = []

for dataset, table_name in TABLES.items():

    if not spark.catalog.tableExists(table_name):
        missing_tables.append(table_name)

if missing_tables:

    raise Exception(
        "Missing Silver source tables: "
        + ", ".join(missing_tables)
    )

print("All Silver source tables exist.")


## 4. Load Silver datasets

TABLES_DF = {}

for dataset, table_name in TABLES.items():

    TABLES_DF[dataset] = spark.table(table_name)

    print(
        dataset,
        "records:",
        TABLES_DF[dataset].count()
    )

print("All Silver datasets loaded successfully.")


## 5. Define foreign key relationships

RELATIONSHIPS = [

    {
        "relationship": "Customers to Orders",
        "parent_dataset": "customers",
        "parent_key": "customer_id",
        "child_dataset": "orders",
        "child_key": "customer_id"
    },

    {
        "relationship": "Orders to Order Items",
        "parent_dataset": "orders",
        "parent_key": "order_id",
        "child_dataset": "order_items",
        "child_key": "order_id"
    },

    {
        "relationship": "Products to Order Items",
        "parent_dataset": "products",
        "parent_key": "product_id",
        "child_dataset": "order_items",
        "child_key": "product_id"
    },

    {
        "relationship": "Orders to Payments",
        "parent_dataset": "orders",
        "parent_key": "order_id",
        "child_dataset": "payments",
        "child_key": "order_id"
    },

    {
        "relationship": "Orders to Shipments",
        "parent_dataset": "orders",
        "parent_key": "order_id",
        "child_dataset": "shipments",
        "child_key": "order_id"
    },

    {
        "relationship": "Products to Inventory",
        "parent_dataset": "products",
        "parent_key": "product_id",
        "child_dataset": "inventory",
        "child_key": "product_id"
    }
]

print("Configured relationships:", len(RELATIONSHIPS))


## 6. Define relationship analysis function

def analyze_relationship(config):

    relationship = config["relationship"]

    parent_dataset = config["parent_dataset"]
    parent_key = config["parent_key"]

    child_dataset = config["child_dataset"]
    child_key = config["child_key"]

    parent_df = TABLES_DF[parent_dataset]
    child_df = TABLES_DF[child_dataset]

    # Validate source columns.

    if parent_key not in parent_df.columns:

        raise Exception(
            f"Parent key {parent_key} missing in {parent_dataset}"
        )

    if child_key not in child_df.columns:

        raise Exception(
            f"Child key {child_key} missing in {child_dataset}"
        )

    # Total records.

    parent_records = parent_df.count()
    child_records = child_df.count()

    # Standardize parent key.

    parent_keys_df = (
        parent_df
        .select(
            F.trim(
                F.col(parent_key).cast("string")
            ).alias("_parent_key")
        )
    )

    # Exclude null and blank parent keys.

    valid_parent_keys_df = (
        parent_keys_df
        .filter(
            F.col("_parent_key").isNotNull()
            & (F.col("_parent_key") != "")
        )
        .distinct()
    )

    unique_parent_keys = valid_parent_keys_df.count()

    # Standardize child key.

    child_keys_df = (
        child_df
        .select(
            F.when(
                F.col(child_key).isNull()
                | (
                    F.trim(
                        F.col(child_key).cast("string")
                    ) == ""
                ),
                F.lit(None).cast("string")
            )
            .otherwise(
                F.trim(
                    F.col(child_key).cast("string")
                )
            )
            .alias("_child_key")
        )
    )

    # Count null and blank child foreign keys.

    null_child_foreign_keys = (
        child_keys_df
        .filter(F.col("_child_key").isNull())
        .count()
    )

    # Keep only valid child foreign keys.

    valid_child_keys_df = (
        child_keys_df
        .filter(F.col("_child_key").isNotNull())
    )

    non_null_child_records = valid_child_keys_df.count()

    # Find child references without matching parent keys.

    orphan_child_df = (
        valid_child_keys_df.alias("child")
        .join(
            valid_parent_keys_df.alias("parent"),
            F.col("child._child_key")
            == F.col("parent._parent_key"),
            "left_anti"
        )
    )

    orphan_child_records = orphan_child_df.count()

    # Find child references with matching parent keys.

    matched_child_df = (
        valid_child_keys_df.alias("child")
        .join(
            valid_parent_keys_df.alias("parent"),
            F.col("child._child_key")
            == F.col("parent._parent_key"),
            "inner"
        )
    )

    referenced_child_records = matched_child_df.count()

    # Count distinct parent keys referenced by children.

    referenced_parent_keys_df = (
        matched_child_df
        .select(
            F.col("child._child_key").alias("_parent_key")
        )
        .distinct()
    )

    referenced_parent_records = (
        referenced_parent_keys_df.count()
    )

    # Find parents without child records.

    parents_without_children_df = (
        valid_parent_keys_df.alias("parent")
        .join(
            referenced_parent_keys_df.alias("referenced"),
            on="_parent_key",
            how="left_anti"
        )
    )

    parents_without_children = (
        parents_without_children_df.count()
    )

    # Reference match rate.

    if non_null_child_records > 0:

        reference_match_rate = round(
            referenced_child_records
            / non_null_child_records * 100,
            2
        )

    else:

        reference_match_rate = 100.0

    # Parent usage rate.

    if unique_parent_keys > 0:

        parent_usage_rate = round(
            referenced_parent_records
            / unique_parent_keys * 100,
            2
        )

    else:

        parent_usage_rate = 0.0

    # Child cardinality distribution.

    child_distribution_df = (
        valid_child_keys_df
        .groupBy("_child_key")
        .agg(
            F.count("*").alias("child_record_count")
        )
        .withColumnRenamed(
            "_child_key",
            child_key
        )
    )

    # Cardinality statistics.

    cardinality_stats = (
        child_distribution_df
        .agg(

            F.count("*").alias("referenced_key_count"),

            F.avg(
                "child_record_count"
            ).alias("average_children_per_key"),

            F.max(
                "child_record_count"
            ).alias("maximum_children_per_key"),

            F.min(
                "child_record_count"
            ).alias("minimum_children_per_key")
        )
        .first()
    )

    average_children_per_key = (
        float(cardinality_stats["average_children_per_key"])
        if cardinality_stats["average_children_per_key"] is not None
        else 0.0
    )

    maximum_children_per_key = (
        int(cardinality_stats["maximum_children_per_key"])
        if cardinality_stats["maximum_children_per_key"] is not None
        else 0
    )

    minimum_children_per_key = (
        int(cardinality_stats["minimum_children_per_key"])
        if cardinality_stats["minimum_children_per_key"] is not None
        else 0
    )

    # Relationship result.

    result = {

        "relationship": relationship,

        "parent_dataset": parent_dataset,
        "parent_key": parent_key,

        "child_dataset": child_dataset,
        "child_key": child_key,

        "parent_records": int(parent_records),

        "unique_parent_keys": int(unique_parent_keys),

        "child_records": int(child_records),

        "null_child_foreign_keys": int(
            null_child_foreign_keys
        ),

        "non_null_child_records": int(
            non_null_child_records
        ),

        "orphan_child_records": int(
            orphan_child_records
        ),

        "referenced_child_records": int(
            referenced_child_records
        ),

        "referenced_parent_records": int(
            referenced_parent_records
        ),

        "parents_without_children": int(
            parents_without_children
        ),

        "reference_match_rate": float(
            reference_match_rate
        ),

        "parent_usage_rate": float(
            parent_usage_rate
        ),

        "average_children_per_key": float(
            average_children_per_key
        ),

        "maximum_children_per_key": int(
            maximum_children_per_key
        ),

        "minimum_children_per_key": int(
            minimum_children_per_key
        ),

        "run_id": RUN_ID,

        "checked_at": CHECKED_AT
    }

    return result, child_distribution_df


## 7. Analyze all relationships

relationship_results = []
cardinality_dataframes = []

for config in RELATIONSHIPS:

    print("=" * 65)
    print("Analyzing:", config["relationship"])

    result, distribution_df = analyze_relationship(config)

    relationship_results.append(result)

    # Standardize all foreign key columns.

    cardinality_df = (
        distribution_df
        .select(
            F.col(config["child_key"])
            .cast("string")
            .alias("child_key_value"),

            F.col("child_record_count")
            .cast("long")
            .alias("child_record_count")
        )

        .withColumn(
            "child_key",
            F.lit(config["child_key"])
        )

        .withColumn(
            "relationship",
            F.lit(config["relationship"])
        )

        .withColumn(
            "parent_dataset",
            F.lit(config["parent_dataset"])
        )

        .withColumn(
            "child_dataset",
            F.lit(config["child_dataset"])
        )

        .withColumn(
            "run_id",
            F.lit(RUN_ID)
        )

        .select(
            "child_key_value",
            "child_record_count",
            "child_key",
            "relationship",
            "parent_dataset",
            "child_dataset",
            "run_id"
        )
    )

    cardinality_dataframes.append(cardinality_df)

    print("Parent records:", result["parent_records"])
    print("Unique parent keys:", result["unique_parent_keys"])
    print("Child records:", result["child_records"])
    print("Null child foreign keys:", result["null_child_foreign_keys"])
    print("Orphan records:", result["orphan_child_records"])
    print("Matched child records:", result["referenced_child_records"])
    print("Reference match rate:", result["reference_match_rate"])
    print("Parent usage rate:", result["parent_usage_rate"])

print("=" * 65)
print("All relationships analyzed.")


## 8. Create relationship report DataFrame

if len(relationship_results) != 6:

    raise Exception(
        f"Expected 6 relationships, found {len(relationship_results)}"
    )

relationship_report_df = spark.createDataFrame(
    relationship_results
)

print("Relationship report schema:")

relationship_report_df.printSchema()

print(
    "Relationship report records:",
    relationship_report_df.count()
)

display(
    relationship_report_df.orderBy("relationship")
)


## 9. Combine cardinality reports

EXPECTED_COLUMNS = [
    "child_key_value",
    "child_record_count",
    "child_key",
    "relationship",
    "parent_dataset",
    "child_dataset",
    "run_id"
]

if len(cardinality_dataframes) != 6:

    raise Exception(
        "Expected six cardinality DataFrames."
    )

for index, df in enumerate(cardinality_dataframes):

    if df.columns != EXPECTED_COLUMNS:

        print("Schema mismatch at index:", index)
        df.printSchema()

        raise Exception(
            f"Unexpected cardinality schema: {df.columns}"
        )

print("All six cardinality schemas are identical.")

cardinality_report_df = cardinality_dataframes[0]

for next_df in cardinality_dataframes[1:]:

    cardinality_report_df = (
        cardinality_report_df.unionByName(
            next_df,
            allowMissingColumns=False
        )
    )

print("Combined cardinality schema:")

cardinality_report_df.printSchema()

print(
    "Total cardinality records:",
    cardinality_report_df.count()
)

display(
    cardinality_report_df.orderBy(
        "relationship",
        F.desc("child_record_count")
    )
)


## 10. Validate relationship results

relationship_count = relationship_report_df.count()

cardinality_count = cardinality_report_df.count()

if relationship_count != 6:

    raise Exception(
        f"Expected 6 relationship records, found {relationship_count}"
    )

if cardinality_count == 0:

    raise Exception(
        "Cardinality report is empty."
    )

# Check that all expected relationship names are present.

expected_relationship_names = {
    config["relationship"]
    for config in RELATIONSHIPS
}

actual_relationship_names = {
    row["relationship"]
    for row in (
        relationship_report_df
        .select("relationship")
        .distinct()
        .collect()
    )
}

if expected_relationship_names != actual_relationship_names:

    raise Exception(
        "Relationship coverage validation failed."
    )

print("Relationship validation passed.")
print("Relationship records:", relationship_count)
print("Cardinality records:", cardinality_count)


## 11. Save foreign key relationship report

(
    relationship_report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RELATIONSHIP_REPORT_TABLE)
)

print("Relationship report saved successfully.")

print("Table:", RELATIONSHIP_REPORT_TABLE)

saved_relationship_count = (
    spark.table(RELATIONSHIP_REPORT_TABLE).count()
)

print(
    "Saved relationship records:",
    saved_relationship_count
)

if saved_relationship_count != 6:

    raise Exception(
        "Relationship report save validation failed."
    )


## 12. Save foreign key cardinality report

(
    cardinality_report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CARDINALITY_REPORT_TABLE)
)

print("Cardinality report saved successfully.")

print("Table:", CARDINALITY_REPORT_TABLE)

saved_cardinality_count = (
    spark.table(CARDINALITY_REPORT_TABLE).count()
)

print(
    "Saved cardinality records:",
    saved_cardinality_count
)

if saved_cardinality_count != cardinality_count:

    raise Exception(
        "Cardinality report save validation failed."
    )


## 13. Analyze parent records without children

parents_without_children_summary_df = (
    relationship_report_df
    .select(
        "relationship",
        "parent_dataset",
        "parent_key",
        "unique_parent_keys",
        "referenced_parent_records",
        "parents_without_children"
    )
    .orderBy(
        F.desc("parents_without_children")
    )
)

display(
    parents_without_children_summary_df
)


## 14. Final relationship health summary

final_summary_df = (
    relationship_report_df
    .agg(

        F.count("*").alias("total_relationships"),

        F.sum("parent_records").alias(
            "total_parent_records"
        ),

        F.sum("child_records").alias(
            "total_child_records"
        ),

        F.sum("null_child_foreign_keys").alias(
            "total_null_child_foreign_keys"
        ),

        F.sum("orphan_child_records").alias(
            "total_orphan_child_records"
        ),

        F.sum("parents_without_children").alias(
            "total_parents_without_children"
        ),

        F.avg("reference_match_rate").alias(
            "average_reference_match_rate"
        ),

        F.avg("parent_usage_rate").alias(
            "average_parent_usage_rate"
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "checked_at",
        F.lit(CHECKED_AT)
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("total_orphan_child_records") > 0,
            "ORPHANS_FOUND"
        )
        .when(
            F.col("total_null_child_foreign_keys") > 0,
            "NULL_FOREIGN_KEYS_FOUND"
        )
        .otherwise("HEALTHY")
    )
)

display(final_summary_df)


## 15. Verify saved tables

required_output_tables = [
    RELATIONSHIP_REPORT_TABLE,
    CARDINALITY_REPORT_TABLE
]

for table_name in required_output_tables:

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Output table was not created: {table_name}"
        )

    print(
        table_name,
        "->",
        spark.table(table_name).count(),
        "records"
    )


## 16. Final validation

final_relationship_count = (
    spark.table(RELATIONSHIP_REPORT_TABLE).count()
)

final_cardinality_count = (
    spark.table(CARDINALITY_REPORT_TABLE).count()
)

if final_relationship_count != 6:

    raise Exception(
        f"Expected 6 relationship rows, found {final_relationship_count}"
    )

if final_cardinality_count != cardinality_count:

    raise Exception(
        "Final cardinality record count does not match."
    )

print("=" * 70)
print("FOREIGN KEY RELATIONSHIP ANALYSIS COMPLETED")
print("=" * 70)

print("Run ID:", RUN_ID)

print(
    "Relationship report:",
    RELATIONSHIP_REPORT_TABLE
)

print(
    "Relationship rows:",
    final_relationship_count
)

print(
    "Cardinality report:",
    CARDINALITY_REPORT_TABLE
)

print(
    "Cardinality rows:",
    final_cardinality_count
)

print("=" * 70)


## 17. Display saved relationship report

display(
    spark.table(RELATIONSHIP_REPORT_TABLE)
    .orderBy("relationship")
)


## 18. Display saved cardinality report

display(
    spark.table(CARDINALITY_REPORT_TABLE)
    .orderBy(
        "relationship",
        F.desc("child_record_count")
    )
)

print("NOTEBOOK 06.3 FINISHED SUCCESSFULLY.")



## Understand Relationship Cardinality:


### One-to-Many (1:N) Relationship:


A `one-to-many` relationship means **one record in a parent table** can be associated with **multiple records in a child table**.


### Example: One Customer → Multiple Orders



```text
             CUSTOMERS TABLE
                (PARENT)

             ┌───────────┐
             │ CUST001   │
             └───────────┘
                   │
          ┌────────┼────────┐
          │        │        │
          ▼        ▼        ▼
       ORD001   ORD002   ORD003
          │        │        │
          └────────┴────────┘
               ORDERS TABLE
                  (CHILD)
```

**Explanation:** One customer (CUST001) can place multiple orders (ORD001, ORD002, ORD003).


### One-to-Many Relationships in E-Commerce:


| Parent Table | Child Table | Relationship |
|---|---|---|
| Customers | Orders | One customer → Many orders |
| Orders | Order Items | One order → Many order items |
| Products | Order Items | One product → Many order items |
| Orders | Payments | One order → Multiple payment records |
| Products | Inventory Snapshots | One product → Multiple inventory snapshots |


### Real-Time Examples:


1. **Customer → Orders:** A customer places three different orders.
2. **Order → Order Items:** One order contains multiple products.
3. **Product → Order Items:** The same product appears in multiple orders.
4. **Order → Payments:** One order can have multiple payment transactions or attempts.
5. **Product → Inventory:** One product has stock records across multiple warehouses and snapshot dates.


### Important Interview Concepts:


| Concept | Meaning |
|---|---|
| Cardinality | Defines how many records can be associated between tables. |
| One-to-One (1:1) | One record relates to one record. |
| One-to-Many (1:N) | One parent record relates to multiple child records. |
| Many-to-Many (M:N) | Multiple records in one table relate to multiple records in another table. |


**Interview Answer:**

- "Relationship cardinality defines the number of associations between records in two tables. 
- In my E-Commerce Lakehouse project, Customers and Orders have a one-to-many relationship because one customer can place multiple orders. 
- This relationship is established using customer_id as a foreign key."